# Прогнозирование ареала пихты для различных сценариев CMIP6

## 1. Подготовка к работе

### 1.1. Импорт библиотек и функций

<div alert class='alert alert-info'>

В ходе работы будут использованы следующие библиотеки и импорты из них: 

- `accuracy_score` и прочие `..._score` – расчёт величины метрик, 
- `geopandas` – для конвертации граничащих массивов однородных точек в полигоны (нужно для создания карты), 
- `json` – для загрузки конфигов, 
- `numpy` – вспомогательная библиотека, необходимая для работы с массивами, 
- `os` – задать рабочую папку, 
- `pandas` – работа с таблицами, 
- `pickle` – сохранение и загрузка разработанных моделей, 
- `plotly.express` – для построения карты, 
- `pyarrow` – для сохранения данных в формате `parquet`, 
- `sys` – для загрузки скриптов, 
- `time` – для отсчёта времени при отображении прогресса в создании карты; 
- `warnings` – предотвратить появление сообщений об ошибках для улучшения читаемости результатов.

</div>

In [ ]:
import os
import pickle
import sys
import time
import warnings

import geopandas as gpd
import json
import numpy as np
import pandas as pd 
import plotly.express as px
import pyarrow as pa
from pyarrow import parquet as pq
from sklearn.metrics import accuracy_score, precision_score, recall_score


### 1.3. Установка значений некоторых опций и констант

In [ ]:
# Нужно, чтобы при отладке не перезапускать ноутбук каждый раз 
# после внесения правок в скрипт. 
%load_ext autoreload
%autoreload 2

In [ ]:
# для улучшения читаемости проекта запретим показывать предупреждения
warnings.filterwarnings("ignore")

In [ ]:
# загрузить конфиг
with open('C:/Users/user/Yandex.Disk/Важные документы/Исходные данные для статей/Моделирование вспышек/Моделирование ареала пихты/Abies_01/config.json', 'r', encoding='utf-8') as f:
    config = json.load(f)

# будем ли мы рассчитывать характеристики погоды
CALCULATE_THE_WHOLE_MAP = config['CALCULATE_THE_WHOLE_MAP']
# имя модели, используемой в работе
MODEL_NAME = config['MODEL_NAME']
# путь к выбранной модели
MODEL_PATH = config['MODEL_PATH']
# загрузить названия файлов с данными по пихте и прочим лесообразователям
NAME_ABIES_FILE = config['NAME_ABIES_FILE']
NAME_OTHERSP_FILE = config['NAME_OTHERSP_FILE']
# адрес корневой папки
ROOT_DIR = config['ROOT_DIR']
# SEED
SEED = config['SEED']

In [ ]:
# нужно для уверенной загрузки скриптов из папки 'scripts'
scripts_path = os.path.join(ROOT_DIR, "scripts")

if scripts_path not in sys.path:
    sys.path.append(scripts_path)


## 2. Обработка полного массива данных о прошлом климате

In [ ]:
# загрузим модель, признанную лучшей, т.е., LightGBM, обученную с метрикой Fbeta при beta=0.6
if CALCULATE_THE_WHOLE_MAP: 
    os.chdir(ROOT_DIR + MODEL_PATH)
    model_for_map = pickle.load(open(MODEL_NAME, 'rb'))
    display(model_for_map)

In [ ]:
if CALCULATE_THE_WHOLE_MAP: 
    # для начала удалим файл 'data.csv' из папки 'map_current', если он существует
    # NOTE: пути здесь и далее подразумевают, что исходные данные лежат в папке 'data', 
    # а результаты сохраняются в 'map_current'. 
    data_path = os.path.join(ROOT_DIR, 'map_current/data.csv')
    if os.path.isfile(data_path): 
        try:
            os.remove(data_path)
            print("Файл 'data.csv' успешно удален из папки 'map_current'")
        except FileNotFoundError:
            print("Файл не найден, удалять нечего")
        except PermissionError:
            print("Нет прав на удаление (или файл открыт в другой программе)")

In [ ]:
if CALCULATE_THE_WHOLE_MAP: 
    from load_weather_completely import load_weather_completely

    # загрузим данные для пихтовых древостоев
    predictions_abies = load_weather_completely(
        abies_file=NAME_ABIES_FILE, 
        other_file=NAME_OTHERSP_FILE, 
        chunk_volume=100000, 
        model=model_for_map, 
        mode='abies'
)

In [ ]:
if CALCULATE_THE_WHOLE_MAP: 
    # загрузим данные для прочих лесообразователей2
    predictions_other = load_weather_completely(
        abies_file=NAME_ABIES_FILE, 
        other_file=NAME_OTHERSP_FILE, 
        chunk_volume=100000, 
        model=model_for_map, 
        mode='other'  
    )

In [ ]:
if CALCULATE_THE_WHOLE_MAP: 
    # HARDCODED: предполагается, что сначала идут метки для пихтовых древостоев, 
    # а за ними – для прочих. 
    # создадим список меток реальных (по ДЗЗ) древостоев ('1': пихта; '2': прочие)
    real = [*np.ones(len(predictions_abies)), *np.zeros(len(predictions_other))]
    # создадим список меток результатов работы модели
    predicted = [*predictions_abies, *predictions_other]

    # если файл с метриками существует, удалить
    if os.path.isfile(os.path.join(ROOT_DIR, 'map_current/metrics.txt')): 
        os.remove(os.path.join(ROOT_DIR, 'map_current/metrics.txt'))

    # записать файл с метриками модели, использованной для создания карты
    with open(os.path.join(ROOT_DIR, 'map_current/metrics.txt'), mode='w') as f: 
        f.write(f'Исследована модель {MODEL_NAME} \n')
        f.write(f'Путь к модели: {MODEL_PATH} \n\n')
        f.write(f'Значение метрик для полного набора данных: \n')
        f.write(f'    accuracy: {accuracy_score(real, predicted)} \n')
        f.write(f'    precision: {precision_score(real, predicted)} \n')
        f.write(f'    recall: {recall_score(real, predicted)} \n')


In [ ]:
if CALCULATE_THE_WHOLE_MAP: 
    if not os.path.isfile(os.path.join(ROOT_DIR, 'map_current/data.parquet')): 
        df_full = pd.read_csv(os.path.join(ROOT_DIR, 'map_current/data.csv'), engine='pyarrow')
        # Конвертируем DataFrame в таблицу PyArrow напрямую
        table = pa.Table.from_pandas(df_full)

        # Сохраняем в Parquet родным методом pyarrow
        output_parquet = os.path.join(ROOT_DIR, 'map_current/data.parquet')
        pq.write_table(table, output_parquet)


<div class='alert alert-info'>

Код для создания карты написан с помощью Google AI. 

</div>

In [ ]:
if CALCULATE_THE_WHOLE_MAP: 
    # создаем данные
    data = pd.read_csv(os.path.join(ROOT_DIR, 'map_current/data.csv'), header=0)


In [ ]:
if CALCULATE_THE_WHOLE_MAP: 
   # cоздаем регулярную сетку точек в градусах
   lon = np.array(data['point_x'])
   lat = np.array(data['point_y'])
   classes = np.array(data['prognosis_res'])

   # инициализируем исходный GeoDataFrame
   gdf_points = gpd.GeoDataFrame(
      {'class_id': classes},
      geometry=gpd.points_from_xy(lon, lat),
      crs="EPSG:4326"
   )

   # трансформируем каждую точку в полигон-ячейку сетки
   grid_cells = []

   # Заменяем точечную геометрию на полигональную
   gdf_cells = gdf_points.copy()
   # HACK: статичное значение буфера – это плохо, но расчёт для каждой точки 
   # приводит к тому, что объединить их в полигоны не получается. 
   gdf_cells['geometry'] = gdf_points.geometry.buffer(0.01)

   # объединяем смежные полигоны с одинаковым class_id
   gdf_polygons = gdf_cells.dissolve(by='class_id', as_index=False)

   # разделяем несвязанные полигоны (MultiPolygon -> Polygon)
   gdf_polygons = gdf_polygons.explode(index_parts=False).reset_index(drop=True)



In [ ]:
if CALCULATE_THE_WHOLE_MAP: 
    # упрощаем геометрию полигонов для ускорения рендеринга
    gdf_polygons['geometry'] = gdf_polygons['geometry'].simplify(tolerance=0.01, preserve_topology=True)
    # добавляем ID полигонов
    gdf_polygons['id'] = gdf_polygons.index.astype(str)

    # выводим отчёт о начале работы
    print("Шаг 1: Конвертируем полигоны GeoPandas в формат GeoJSON...")
    start_time = time.time()

    # извлекаем интерфейс геометрии отдельно
    geojson_data = gdf_polygons.__geo_interface__

    # выводим отчёт о продолжении работы
    print(f"Конвертация завершена за {time.time() - start_time:.2f} сек.")
    print("Шаг 2: Передаем данные в Plotly и генерируем карту...")

    # строим карту
    fig = px.choropleth(
        # данные
        gdf_polygons, 
        # геоданные
        geojson=geojson_data, 
        # метки цвета (= результат прогнозирования)
        color='class_id', 
        # метки полигонов
        locations='id', 
        # цветовая карта
        color_discrete_map={
            'TP': 'purple', 
            'TN': 'green', 
            'FP': 'magenta', 
            'FN': 'yellow'        
        }, 
        # картографическая проекция
        projection="azimuthal equal area", 
        # вид карты
        scope='world',
        # заголовок
        title=f"Predicted current vegetation; model name is {MODEL_NAME}"    
        )
    # отображение региона исследования
    fig.update_geos(
        # принудительно включаем отображение только нужного региона
        fitbounds="locations"
    )
    # удаление белых линий на стыках полигонов (там, где между ними возникают зазоры)
    fig.update_traces(marker_line_width=0)

    # отчёт о начале рендеринга
    print("Карта успешно построена! Рендеринг в браузере...")
    # вывод карты на дисплей
    fig.show()
    # сохранение карты на диск
    fig.write_html(os.path.join(ROOT_DIR, 'map_current/abies_range_interactive_map.html'))


In [ ]:
if CALCULATE_THE_WHOLE_MAP: 
    del real, predicted, predictions_abies, predictions_other

<div class='alert alert-warning'>

Карта не будет создана, т.к. при таких объёмах данных средствами Python удаётся получить только карту огромного объёма (св. 1 Гб). Отдам Саше Шушпанову, пусть развлекается. 

</div>

## 3. Моделирование для будущего климата

<div class='alert alert-info'>

Данные скачаны [вот из этого источника](https://cds.climate.copernicus.eu/datasets/projections-cmip6). База требует ссылки в этом формате: *Copernicus Climate Change Service, Climate Data Store, (2021): CMIP6 climate projections. Copernicus Climate Change Service (C3S) Climate Data Store (CDS). DOI: 10.24381/cds.c866074c (Accessed on DD-MMM-YYYY)*. 

Координаты данных: 79 вд; 47 сш; 101 вд; 65 сш.

Какие именно модели дали данные для каждого из сценариев, отмечено в таблицах ниже. Там же указаны номера литературных источников на эти модели. 

</div>


Содержание влаги в верхнем слое почвы (`soil_water`); апрель: 

| Модель | Источники данных | SSP1-2.6 | SSP4-3.4 | SSP2-4.5 | SSP4-6.0 | SSP4-7.0 | SSP4-8.5 |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| BCC-CSM2-MR | [2400] | + | | + | | + | + |   
| CMCC-CM2-SR5 | [2455] | + | | + | | + | + |   
| CMCC-ESM2 | [2455] | + | | + | | | + |  
| EC-Earth3-CC | [2407] | | | + | | | + |  
| GFDL-ECM4 | [2407] | + | | + | | + | + |   
| IPSL-CM6A-LR | [2400] | + | + | + | + | + | + |  
| MIROC-6 | [2400] | + | + | + | + | + | + |  
| MPI-ESM1-2-LR | [2400, 2407] | + | | + | | + | + |  
| TaiECM1 | [2407] | + | | + | | + | + |  



Эвапотранспирация (`evap`); март, апрель, июль, ноябрь: 
| Модель | Источники данных | SSP1-2.6 | SSP4-3.4 | SSP2-4.5 | SSP4-6.0 | SSP4-7.0 | SSP4-8.5 |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| BCC-CSM2-MR | [2453] | + | | + | | + | + | 
| CESM2 | [2407, 2453] | | | + | | + | + |
| CESM2-WACCM | [2453] | | | | | + | + |
| CMCC-CM2-SR5 | [2453] | | | | | + | + |
| CMCC-ESM2 | [2407, 2453] | + | | + | | | + | 
| CRNM-ESM2-1 | [2407] | | + | + | + | + | + | 
| EC-Earth3-CC | [2407] | | | + | | | + | 
| GFDL-ESM4 | [2407] | | | + | | + | + | 
| MPI-ESM1-2-LR | [2407] | | | + | | + | + |
| MRI-ESM2-0 | [2407, 2453] | + | + | + | + | + | + | 
| UKESM1-0-LL | [2407] | + | + | + | | + | + | 



Относительная влажность (rel_hum); март, сентябрь, декабрь: 
| Модель | Источники данных | SSP1-2.6 | SSP4-3.4 | SSP2-4.5 | SSP4-6.0 | SSP4-7.0 | SSP4-8.5 |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| ACCESS-ESM1-5 | [2454] | | | | | | | 
| CESM2 | [2454] | | | | | | |  
| CESM2-WACCM | [2454] | | | | | | |  
| GFDL-CM4 | [2454] | | | | | | |  
| GFDL-ESM4 | [2454] | | | | | | |  
| NorESM2-LM | [2454] | | | | | | |  
| NorESM2-MM | [2454] | | | | | | |  
| UKESM1-0-LL | [2454] | | | | | | | 


<div class='alert alert-warning'>

Сценарий SSP4-6.0 по осадкам выпал полностью: выбранные для работы модели не предоставляют данных по этой характеристике в этом сценарии. Ergo: с SSP4-6.0 не работаем вообще. 

</div>


Осадки (pre); январь, июль: 
| Модель | Источники данных | SSP1-2.6 | SSP4-3.4 | SSP2-4.5 | SSP4-6.0 | SSP4-7.0 | SSP4-8.5 |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| BCC-CSM2-MR | [2374] | + | | + | | + | + |  
| CESM2-WACCM | [2374] | | | | | + | + |  
| EC-Earth3 | [2374] | | + | | | | |  
| FGOALS-f3-L | [2374] | + | + | + | | + | + |  
| HadGEM3-GC31-LL | [2374] | + | + | + | | | + |  
| NorESM2-LM | [2374] | + | | + | | + | + | 
| NorESM2-MM | [2374] | + | | + | | + | + | 

In [ ]:
# from get_scenarios_tmp import get_scenarios_tmp

# get_scenarios_tmp()

In [ ]:
# import zipfile
# import io
# import zipfile
# import xarray as xr


In [ ]:
# feature_folder = 'evap'
# scenario = 'cmip26'
# path_to_feature = os.path.join(ROOT_DIR, 'data/cmip_projections', feature_folder, scenario)
# cmip_sets = [f for f in os.listdir(path_to_feature)]


In [ ]:

# path_to_set = os.path.join(path_to_feature, cmip_sets[0])
# with zipfile.ZipFile(path_to_set, 'r') as zip_ref:
#     # 1. Получаем список всех файлов в архиве
#     all_files = zip_ref.namelist()
    
#     # 2. Находим первый файл, который заканчивается на .nc
#     nc_file_name = next((f for f in all_files if f.endswith('.nc')), None)
    
#     # Проверяем, нашли ли мы файл
#     if nc_file_name:
#         # 3. Читаем найденный файл как байты
#         nc_bytes = zip_ref.read(nc_file_name)
        
#         # 4. Открываем его через xarray
#         with xr.open_dataset(io.BytesIO(nc_bytes)) as dataset:
#             dataset = dataset


In [ ]:
# # 1. Безопасно конвертируем нестандартное время xarray в понятный для pandas формат
# # Это исправит проблему с DatetimeNoLeap раз и навсегда
# if hasattr(dataset, 'indexes') and 'time' in dataset.indexes:
#     try:
#         standard_time = dataset.indexes['time'].to_datetimeindex()
#         dataset = dataset.assign_coords(time=standard_time)
#     except AttributeError:
#         # Если это чистый DataArray или метод to_datetimeindex() не применим
#         pass

# dataset = dataset['evspsbl'].to_dataframe().reset_index()
# dataset['time'] = pd.to_datetime(dataset['time'])
# dataset['year'] = dataset['time'].dt.year
# dataset['month'] = dataset['time'].dt.month
# dataset_pivot = dataset.pivot_table(
#     index=['year', 'lat', 'lon'], 
#     columns='month', 
#     values='evspsbl'
# ).reset_index()


In [ ]:
# dataset_pivot.head()

In [ ]:
# poetry add xarray netcdf4 rioxarray

# import xarray as xr
# import os

# nc_path = os.path.join(ROOT_DIR, 'map_current/climate_data.nc')

# # 1. Открываем NetCDF датасет
# ds = xr.open_dataset(nc_path)
# print(ds) # Покажет структуру: координаты (time, lat, lon) и переменные (например, t2m, tp)

# # 2. Вырезаем конкретный срез по времени или региону (если нужно)
# # Пример: берем данные только за определенный период
# ds_2026 = ds.sel(time='2026-01-01')

# # 3. Конвертируем многомерную сетку в плоский Pandas DataFrame для ML-модели
# # Метод to_dataframe() создаст мультииндекс из координат, который мы сбросим
# df_climate = ds_2026.to_dataframe().reset_index()

# # Теперь у вас есть стандартный DataFrame со столбцами: lat, lon, t2m и т.д.
# print(df_climate.head())
